In [ ]:
import pandas as pd
import numpy as np
import librosa
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

def extract_mfcc(file_path, max_pad_len=174):
    y, sr = librosa.load(file_path, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=30)
    if mfcc.shape[1] < max_pad_len:
        pad_width = max_pad_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
    else:
        mfcc = mfcc[:, :max_pad_len]
    return mfcc

train_folder = 'train'
train_csv_path = 'train.csv'
train_df = pd.read_csv(train_csv_path)

train_features = []
train_labels = []

for index, row in train_df.iterrows():
    file_path = os.path.join(train_folder, row['path'].replace('train/', '').replace('./', ''))
    if os.path.exists(file_path):
        features = extract_mfcc(file_path)
        train_features.append(features)
        train_labels.append(row['label'])
    else:
        print(f"File not found: {file_path}")

train_features = np.array(train_features)
train_labels = np.array(train_labels)

train_features = train_features[..., np.newaxis]

le = LabelEncoder()
train_labels_encoded = le.fit_transform(train_labels)

X_train, X_test, y_train, y_test = train_test_split(train_features, train_labels_encoded, test_size=0.2, random_state=42)

cnn_model = Sequential()
cnn_model.add(Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(X_train.shape[1], X_train.shape[2], 1)))
cnn_model.add(MaxPooling2D((2, 2)))
cnn_model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
cnn_model.add(MaxPooling2D((2, 2)))
cnn_model.add(Flatten())
cnn_model.add(Dense(128, activation='relu'))

cnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

X_train_cnn_features = cnn_model.predict(X_train)
X_test_cnn_features = cnn_model.predict(X_test)

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_cnn_features, y_train)  # y_train should be integer labels

y_pred = rf_model.predict(X_test_cnn_features)

accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest Accuracy after CNN feature extraction: {accuracy}")

test_folder = 'test'
test_csv_path = 'test.csv'
sample_submission_path = 'sample_submission.csv'
test_df = pd.read_csv(test_csv_path)

test_features = []
for index, row in test_df.iterrows():
    file_path = os.path.join(test_folder, row['path'].replace('test/', '').replace('./', ''))
    if os.path.exists(file_path):
        features = extract_mfcc(file_path)
        test_features.append(features)
    else:
        print(f"File not found: {file_path}")

test_features = np.array(test_features)
test_features = test_features[..., np.newaxis]

test_cnn_features = cnn_model.predict(test_features)

test_pred = rf_model.predict_proba(test_cnn_features)

submission_df = pd.read_csv(sample_submission_path)
submission_df['real'] = test_pred[:, le.transform(['real'])[0]]
submission_df['fake'] = test_pred[:, le.transform(['fake'])[0]]
submission_df.to_csv('submission.csv', index=False)

print("Submission file saved successfully.")




